# Recovered beam mismatch parameters (Figure 2)

Reproduces **Figure 2** of the companion data paper: the distributions of the
six differential beam-mismatch parameters recovered from the 106 time trunks
for a single detector pair at 150 GHz.

* fitted coefficients: `params/deproj_beam_params.pkl`
* injected parameters: `params/beamsys/*_150G.txt`
* focal-plane angles used to rotate the injected pointing and ellipticity into
  the local detector frame: `params/fp/`

Set `DEPROJ_DATA_ROOT` to the directory holding `params/`, `maps/`, `mask/`
and `FG/` (see the README).  The notebook also writes a figure to `output/`.


In [ ]:
import os
import pathlib
import pickle

import numpy as np
import matplotlib.pyplot as plt

DATA = pathlib.Path(os.environ.get('DEPROJ_DATA_ROOT', '/sharefs/alicpt/users/doujiazheng/data/deproj'))
OUT = pathlib.Path('output')
OUT.mkdir(exist_ok=True)

FREQ = 150          # 95 or 150 GHz
IDET = 1            # detector-pair index
RAD2DEG = 180. / np.pi

plt.style.use('seaborn-v0_8-ticks')
plt.rc('font', family='sans-serif', size=6)

## Fitted parameters

`deproj_beam_params.pkl` is a dictionary keyed by frequency.  Each entry has
shape `(n_trunk, 6, n_pair)`: 106 time trunks, the six differential modes
`(dg, dx, dy, dsigma, dp, dc)` and 1728 detector pairs.  Pointing `dx, dy` are
stored in radians and `dsigma` in degrees.


In [ ]:
with open(DATA / 'params' / 'deproj_beam_params.pkl', 'rb') as f:
    fitted = np.asarray(pickle.load(f)[FREQ])

dg, dx, dy = fitted[:, 0], fitted[:, 1] * RAD2DEG, fitted[:, 2] * RAD2DEG
dsig, dp, dc = fitted[:, 3], fitted[:, 4], fitted[:, 5]
print('(n_trunk, n_mode, n_pair) =', fitted.shape)

## Injected parameters in the local detector frame

Each `params/beamsys/<mode>_<FREQ>G.txt` file has three columns
`(r, theta, value)` per detector pair, where `r` and `theta` are the
focal-plane coordinates.  The pointing differences and the ellipticity are
defined in the focal-plane frame, so they are rotated by twice the
pair-orientation angle `theta + chi` before comparison with the fitted values.


In [ ]:
fp = {k: np.load(DATA / 'params' / 'fp' / (k + '.npy')) for k in ('theta', 'chi')}
dbl = 2. * (fp['theta'] + fp['chi'])          # twice the pair orientation


def injected(mode):
    """Third column of params/beamsys/<mode>_<FREQ>G.txt."""
    return np.loadtxt(DATA / 'params' / 'beamsys' / ('%s_%dG.txt' % (mode, FREQ)))[:, 2]


dg_in, ds_in = injected('dg'), injected('dsigma')
dx_fp, dy_fp = injected('dx'), injected('dy')
dx_in = np.cos(dbl) * dx_fp + np.sin(dbl) * dy_fp
dy_in = -np.sin(dbl) * dx_fp + np.cos(dbl) * dy_fp

de_mod, de_ori = injected('de_conti'), injected('de_orient')
dp_in = de_mod * np.cos(dbl + 2. * de_ori)
dc_in = de_mod * np.sin(dbl + 2. * de_ori)

## Figure 2: distributions over the 106 time trunks

Blue histogram: recovered values for the selected detector pair.  Red and black
dashed lines mark the injected value and the mean over time trunks.


In [ ]:
def plot_pdf(ax, data, truth, xlabel, first=False, bias=False):
    ax.hist(data, bins=20, alpha=0.5)
    ax.axvline(np.mean(data), ls='--', lw=1., color='k', label='mean')
    ax.axvline(truth, ls='--', lw=1., color='r', label='input')
    if first:
        ax.set_ylabel(r'$N_{\rm time}$')
        ax.legend(loc='best', frameon=True)
    if bias:
        ax.text(0.25, 0.95, r'$\Delta$ = {:.3e}'.format(np.mean(data) - truth),
                ha='left', va='top', transform=ax.transAxes)
    ax.set_xlabel(xlabel)
    ax.tick_params(size=3)


fig, ax = plt.subplots(2, 3, figsize=(4.8, 2.9), dpi=300, tight_layout=True)
plot_pdf(ax[0, 0], dg[:, IDET], dg_in[IDET], r'$\delta g$', first=True)
plot_pdf(ax[0, 1], dx[:, IDET] * 60., dx_in[IDET] * 60., r'$\delta x$ [arcmin]')
plot_pdf(ax[0, 2], dy[:, IDET] * 60., dy_in[IDET] * 60., r'$\delta y$ [arcmin]')
plot_pdf(ax[1, 0], dsig[:, IDET] * 60., ds_in[IDET] * 60., r'$\delta\sigma$ [arcmin]', first=True)
plot_pdf(ax[1, 1], dp[:, IDET], dp_in[IDET], r'$\delta p$', bias=True)
plot_pdf(ax[1, 2], dc[:, IDET], dc_in[IDET], r'$\delta c$')
for a in ax.ravel():
    a.ticklabel_format(style='sci', scilimits=(0, 0), axis='x', useMathText=True)
fig.savefig(OUT / ('beam_params_%dGHz.png' % FREQ), bbox_inches='tight', pad_inches=0.1)

The `dp` panel is offset from the injected value by about
7 x 10^-3.  This bias comes from the intrinsic CMB T-E correlation, which
makes the `dp` template partly correlated with the pair-difference data; it is
estimated from systematics-free simulations and corrected in the analysis.


## Input versus recovered, averaged over time trunks

The same quantities averaged over the 106 time trunks for all 1728 detector
pairs.  `R` is the correlation coefficient and `sigma(Delta)` the scatter of
the residual.


In [ ]:
modes = [r'$\delta g$', r'$\delta x$ [arcmin]', r'$\delta y$ [arcmin]',
         r'$\delta\sigma$ [arcmin]', r'$\delta p$', r'$\delta c$']
truth = [dg_in, dx_in * 60., dy_in * 60., ds_in * 60., dp_in, dc_in]
recovered = [f.mean(axis=0) for f in (dg, dx * 60., dy * 60., dsig * 60., dp, dc)]

fig, ax = plt.subplots(2, 3, figsize=(4.0, 2.9), dpi=300, tight_layout=True)
for a, x, y, label in zip(ax.ravel(), truth, recovered, modes):
    a.plot([x.min(), x.max()], [x.min(), x.max()], 'r-', lw=0.5)
    a.scatter(x, y, s=0.2, c='grey', alpha=0.7)
    a.set_xlabel(label)
    a.tick_params(size=3)
    a.ticklabel_format(style='sci', scilimits=(0, 0), axis='both', useMathText=True)
    a.text(0.05, 0.95, 'R = {:.3f}'.format(np.corrcoef(x, y)[0, 1]),
           ha='left', va='top', transform=a.transAxes)
    a.text(0.95, 0.05, r'$\sigma(\Delta)$ = {:.2e}'.format(np.std(x - y)),
           ha='right', va='bottom', transform=a.transAxes)
ax[0, 0].set_ylabel('recovered')
ax[1, 0].set_ylabel('recovered')
fig.savefig(OUT / ('beam_params_scatter_%dGHz.png' % FREQ), bbox_inches='tight', pad_inches=0.1)